# 07 · Circuits and laws with exege

Two papers, two tools, both run on the real ViT through exege's adapter (`vit-interp`). Nothing here is ViT-specific: exege only sees a system that can be **patched**.

* **Seeing Through Circuits (Vi-CD):** which **edges** (head → head, head → MLP, … → output) does a behaviour run through? We patch edges from a counterfactual run and keep the ones whose loss moves the metric. The graph has 57 senders, 17 receivers and **513 edges**.
* **LAWFUL:** does the model obey a **law** (here barotropic Rossby dispersion), and which components carry that agreement (sufficiency / necessity)?

Command-line equivalents are at the end: `exege circuits …` and `exege laws …`.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
try:
    import vit_interp
except ImportError:  # not installed: use this checkout, $VIT_INTERP_REPO, or the shared one
    for repo in (Path.cwd().parent, Path(os.environ.get("VIT_INTERP_REPO", "/home/ac.ngmahfouz/vit-interp"))):
        if (repo / "vit_interp").is_dir():
            sys.path.insert(0, str(repo)); break
import numpy as np, torch, matplotlib.pyplot as plt
from vit_interp import playground as pg
em, meta = pg.load()      # or pg.load("/path/to/checkpoint.pt"); $PLAYGROUND_CKPT also works
FAST = pg.FAST            # PLAYGROUND_FAST=1 -> small settings (CPU tests)

In [ ]:
from exege.circuits import open_patchable, field_mean, attribute, prune, faithfulness_curve, core, jaccard
from exege.laws import Rossby, Envelope, measure, deviations, search, components
CKPT = os.environ.get("PLAYGROUND_CKPT") or str(pg.default_checkpoint())
system = open_patchable(CKPT, "vit-interp", year=2010)
g = system.graph(); grid = system.grid()
print(len(g.senders), "senders,", len(g.receivers), "receivers,", g.n_edges, "edges")

## A behaviour and its counterfactual

The behaviour is the mean 6-h **V_3** (jet-level meridional wind) over the North Pacific. Its counterfactual is the same metric 5 days later (another weather state). Vi-CD inpaints the object out of an image; for an emulator *which* counterfactual you choose is the scientific question, so try others below.

In [ ]:
metric = field_mean(grid, "V_3", where=grid.within(45, 180, 2000))
base = system.states(["2010-01-15T00"])
source = system.states(["2010-01-20T00"])

## Score every edge (EAP: one gradient), then prune (one patched step per edge)

In [ ]:
a = attribute(system, base, source, metric, method="eap")
print(f"clean {a.clean:.3f}, counterfactual {a.corrupt:.3f}")
import pandas as pd
pd.DataFrame(a.ranked(15), columns=["edge", "score (metric change if this edge alone is patched)"])

In [ ]:
c = prune(system, base, source, metric, tau=0.05 if FAST else 0.02)
print(f"{c.n_edges} of {g.n_edges} edges recover {c.recovery:.1%} of the clean-vs-counterfactual gap")
print("components in the circuit:", " ".join(c.components()))

Does EAP's ranking find faithful circuits? Keep the top-k edges by EAP score and patch the rest (Vi-CD's first experiment):

In [ ]:
sizes = [5, 20, 60, 150] if FAST else [5, 10, 20, 40, 60, 100, 200, 513]
curve = faithfulness_curve(system, base, source, metric, a, sizes)
plt.plot(curve.sizes, curve.recoveries, "o-"); plt.axhline(1, color="k", lw=0.5)
plt.xscale("log"); plt.xlabel("edges kept (top-k by EAP)"); plt.ylabel("recovery"); plt.title("faithfulness vs sparsity");

## Rossby waves: LAWFUL

Plant a small, balanced Rossby wave of zonal wavenumber s = 2…14 at 45°N on several real states, take one step, and read the wave's frequency off the model's response. Compare with ω − kŪ = −βk/(k²+l²).

* `score="eq10"` is LAWFUL's misfit score: 1 means the model matches the law, and negative means it is worse than the law's mean.
* `score="correlation"` asks only whether the model follows the law's **shape** across s.

In [ ]:
law = Rossby(u=("U_2", "U_3", "U_4"), v=("V_2", "V_3", "V_4"), observe_field="V_3", ps="PS",
             band=Envelope(45, 10), wavenumbers=range(2, 9) if FAST else range(2, 15))
X = system.states([0, 120] if FAST else [0, 60, 120, 180, 240, 300, 360, 420])
r10 = measure(system, X, law); rc = measure(system, X, law, score="correlation")
print(f"M (eq10) = {r10.M:.2f}    M (correlation) = {rc.M:.3f}")
df = pd.DataFrame(r10.rows())
plt.plot(df.alpha, df.model, "o-", label="ViT"); plt.plot(df.alpha, df.law, "s--", label="barotropic theory")
plt.xlabel("zonal wavenumber s"); plt.ylabel("intrinsic frequency ω − kŪ [rad/day]"); plt.legend();

Which blocks carry the shape? Sufficiency (`d_suf` small = this block alone reproduces it) and necessity (`d_nec` large = the rest can't do without it), on the correlation score:

In [ ]:
rows = []
for L in ([4, 5, 6] if FAST else range(8)):
    comps = [f"a{L}.h{h}" for h in range(6)] + [f"m{L}"]
    d = deviations(system, X, law, comps, score="correlation")
    rows.append({"block": L, "M_suf": d.M_suf, "M_nec": d.M_nec, "d_suf": d.d_suf, "d_nec": d.d_nec})
pd.DataFrame(rows).round(3)

LAWFUL's greedy search for τ-consistent circuits (nested, minimal). Each candidate costs one patched sweep of the family, so search **whole blocks** first (8 candidates, a couple of minutes on a GPU), then single heads inside the blocks it picks (`candidates=[...]`).

In [ ]:
from exege.laws import blocks
if not FAST:
    s = search(system, system.states([0, 120, 240, 360]), law, groups=blocks(system),
               taus=(0.5, 0.7, 0.9), score="correlation", controls=[system.states([60, 180, 300, 420])])
    for circ in s.circuits:
        d = circ.deviations
        print(f"tau {circ.tau}: {len(circ.components)} components  d_suf {d.d_suf:.3f}  d_nec {d.d_nec:.3f}"
              f"  control d_suf {[round(x, 3) for x in d.to_dict().get('d_suf_random', [])]}")
        print("   ", " ".join(circ.units or circ.components))

## The same from a terminal

```console
$ exege circuits attribute --adapter vit-interp --source $CKPT --field V_3 --lat 45 --lon 180 \
    --base 2010-01-15T00 --source-starts 2010-01-20T00 --method eap
$ exege circuits prune ... --tau 0.02 --out circuit.json
$ exege laws rossby --adapter vit-interp --source $CKPT --u U_2,U_3,U_4 --v V_2,V_3,V_4 \
    --observe V_3 --ps PS --phi0 45 --starts 0,60,120,180 --score correlation --search --by-block
```

## Try this

1. **Counterfactuals.** Swap the source for the climatology, or for the same day with the trough removed (`system.perturb`), or for a day 6 hours later. Do circuits change?
2. **Stable cores.** Prune for 5 different January days, then `core([...])`. Which edges are always there? (Vi-CD RQ2)
3. **Regions.** Run circuits for tropics vs midlatitudes vs polar boxes and compare with `jaccard`.
4. **Steering level.** `Rossby(..., u_bar="U_5")`: does a lower steering wind close the gap with theory?
5. **Hemispheres and seasons.** `Envelope(-45, 10)`, and July states (`system.states(["2010-07-15T00", ...])`). β is the same, Ū is not.
6. **Errors** (the Nov 4 paper): pick a metric that is the model's *error* (ViT minus E3SM) in a region, and find the circuit that produces it.